# 🏠 Day 14 · Borgesian map: every building in Uzbekistan

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2014%20Borgesian%20map/tutorial/day14_every_building.ipynb)

In this notebook you will:
1. ask questions to a table of **7 million buildings** with **DuckDB** and SQL,
2. draw **every building in Uzbekistan** as one dot, and watch the country appear,
3. zoom in: Tashkent, then the houses of Bukhara's old town,
4. find where the map is **not** the territory,
5. calculate how big a 1:1 map would be, like in Borges' story.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

**DuckDB** is a small, very fast database that runs inside Python. You ask it questions in **SQL**, the language of databases. It can read files straight from the internet.

In [ ]:
import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/NodiraTillayeva/30DayMapChallengeUzbekistan/main/2026/Day%2014%20Borgesian%20map/tutorial/data/"

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial;")
print("Ready! ✅")

## Step 1 · Load 7 million buildings

`uz_buildings.parquet` (40 MB) has one row per building: its **region**, the **lon** and **lat** of its centre, its **area** in m² and its **source**. We copy it into DuckDB as a table called `buildings`. It takes about 20 seconds.

In [ ]:
con.sql(f"CREATE OR REPLACE TABLE buildings AS SELECT * FROM read_parquet('{BASE}uz_buildings.parquet')")
con.sql("SELECT * FROM buildings LIMIT 5")

## Step 2 · Ask questions with SQL

A SQL question has a simple shape: `SELECT` what you want `FROM` which table, maybe `WHERE` something is true, `GROUP BY` something, `ORDER BY` something.

**How many buildings, and who drew them?**

In [ ]:
con.sql("""
    SELECT src AS source, count(*) AS buildings
    FROM buildings
    GROUP BY src
""")

**How many in each region?**

In [ ]:
con.sql("""
    SELECT region, count(*) AS buildings, round(avg(area_m2)) AS average_m2
    FROM buildings
    GROUP BY region
    ORDER BY buildings DESC
""")

**The ten biggest buildings.** Copy a `lat, lon` pair into Google Maps to see what they are!

In [ ]:
con.sql("""
    SELECT region, area_m2, lat, lon
    FROM buildings
    ORDER BY area_m2 DESC
    LIMIT 10
""")

## Step 3 · Every building in Uzbekistan

Now the Borgesian part: one tiny black dot for each of the 7 million buildings. No borders, no rivers, no roads: only houses. Drawing takes about 30 seconds.

In [ ]:
pts = con.sql("SELECT lon, lat FROM buildings").df()

fig, ax = plt.subplots(figsize=(16, 9))
ax.scatter(pts["lon"], pts["lat"], s=0.02, c="black", linewidths=0)
ax.set_aspect(1.3)          # 1 / cos(40°): makes the map look right at Uzbekistan's latitude
ax.set_axis_off()
plt.show()

## Step 4 · Zoom in: Tashkent

The same dots, only for `Tashkent City`, a little bigger.

In [ ]:
tash = con.sql("SELECT lon, lat FROM buildings WHERE region = 'Tashkent City'").df()
print(len(tash), "buildings")

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(tash["lon"], tash["lat"], s=0.3, c="black", linewidths=0)
ax.set_aspect(1.3)
ax.set_axis_off()
plt.show()

## Step 5 · Zoom in more: the houses of Bukhara

`bukhara_old_town.parquet` has the full **shapes** (footprints) of the 4,100 buildings in Bukhara's old town. Now each building is not a dot but a polygon. The colour shows its size.

In [ ]:
bukhara = gpd.read_parquet(BASE + "bukhara_old_town.parquet")
print(len(bukhara), "buildings")

ax = bukhara.to_crs(32641).plot(column="area_m2", cmap="copper_r", vmax=600, figsize=(12, 10))
ax.set_axis_off()
plt.show()

## Step 6 · Where the map is not the territory

Look at the big map from Step 3 again. In the middle of the **Ferghana Valley** there's an empty belt with straight edges, and **Khorezm** looks almost empty. People live there, of course! These are **gaps in the data**: the AI that traced buildings from satellite images didn't cover those places, and nobody has drawn them in OpenStreetMap yet.

Let's count buildings in squares of 0.2° × 0.2° in the Ferghana Valley. The zeros in the middle are the gap.

In [ ]:
con.sql("""
    SELECT floor(lat * 5) / 5 AS lat, floor(lon * 5) / 5 AS lon, count(*) AS n
    FROM buildings
    WHERE lon BETWEEN 70.6 AND 72.6 AND lat BETWEEN 40.2 AND 41.2
    GROUP BY ALL
""").df().pivot(index="lat", columns="lon", values="n").fillna(0).astype(int).sort_index(ascending=False)

## Step 7 · How big would a 1:1 map be?

In Borges' story, the mapmakers drew a map as big as the empire. Let's do the maths for our buildings.

In [ ]:
total_m2 = con.sql("SELECT sum(area_m2) FROM buildings").fetchone()[0]
uzbekistan_m2 = 448_978e6        # Uzbekistan: 448,978 km²
a4_m2 = 0.210 * 0.297

print("All roofs together:", round(total_m2 / 1e6), "km²")
print("That's", round(100 * total_m2 / uzbekistan_m2, 2), "% of Uzbekistan")
print("A 1:1 map of them needs", f"{total_m2 / a4_m2:,.0f}", "A4 sheets")

## Step 8 · Save your map

The big map again, sharper, saved as **every_building.png** and downloaded.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9), facecolor="white")
ax.scatter(pts["lon"], pts["lat"], s=0.02, c="black", linewidths=0)
ax.set_aspect(1.3)
ax.set_axis_off()
plt.savefig("every_building.png", dpi=300, bbox_inches="tight")

from google.colab import files
files.download("every_building.png")

## Extra · Ask the cloud directly (slow)

Our file was made from **Overture Maps**, an open map of the whole world. DuckDB can also ask Overture directly for any box, for example your town. It reads from a huge collection of files, so it takes **5–10 minutes**.

Change `RUN_LIVE` to `True` and the box (`west, south, east, north`) to your town, then run the cell.

In [ ]:
RUN_LIVE = False
west, south, east, north = 59.58, 42.43, 59.65, 42.49     # Nukus city centre

if RUN_LIVE:
    con.sql("SET s3_region='us-west-2'")
    town = con.sql(f"""
        SELECT sources[1].dataset AS source, ST_AsWKB(geometry) AS geometry
        FROM read_parquet('s3://overturemaps-us-west-2/release/2026-10-21.0/theme=buildings/type=building/*', hive_partitioning=1)
        WHERE bbox.xmin > {west} AND bbox.xmax < {east} AND bbox.ymin > {south} AND bbox.ymax < {north}
    """).df()
    town = gpd.GeoDataFrame(town, geometry=gpd.GeoSeries.from_wkb(town["geometry"].apply(bytes)), crs=4326)
    print(len(town), "buildings")
    town.plot(column="source", legend=True, figsize=(10, 10)).set_axis_off()
    plt.show()

## 🎉 Done!

Share your map with **#30DayMapChallenge**.

**Credits:** buildings © Overture Maps Foundation, release 2026-10-21.0 (ODbL), including © OpenStreetMap contributors and Microsoft ML Buildings.